In [1]:
!pip install textblob
!python -m textblob.download_corpora

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package conll2000 to /root/nltk_data...
[nltk_data]   Unzipping corpora/conll2000.zip.
[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Unzipping corpora/movie_reviews.zip.
Finished.


In [2]:
!pip install emoji spacy
!python -m spacy download en_core_web_sm

import re
import nltk
import emoji
import spacy
from nltk.corpus import stopwords
from textblob import TextBlob

nltk.download('stopwords')

nlp = spacy.load("en_core_web_sm")

class AdvancedNLPPreprocessor:

    def __init__(self, config=None):
        self.stop_words = set(stopwords.words("english"))

        # 🔥 Slang Dictionary
        self.slang_dict = {
            "lol": "laugh",
            "omg": "oh my god",
            "idk": "i do not know",
            "brb": "be right back",
            "bro": "brother"
        }

        # 🔥 Config (Pipeline Control)
        self.config = config if config else {
            "lower": True,
            "emoji": True,
            "url": True,
            "special_char": True,
            "repeat": True,
            "slang": True,
            "tokenize": True,
            "stopwords": True,
            "lemmatize": True,
            "negation": True,
            "spell": False  # keep False (slow)
        }

    # Lowercase
    def to_lower(self, text):
        return text.lower()

    # Remove URLs
    def remove_urls(self, text):
        return re.sub(r"http\S+|www\S+", "", text)

    # Remove special chars
    def remove_special_chars(self, text):
        return re.sub(r"[^a-zA-Z\s]", "", text)

    # Normalize repeated characters
    def normalize_repeated_chars(self, text):
        return re.sub(r'(.)\1{2,}', r'\1\1', text)

    # Slang normalization
    def normalize_slang(self, text):
        words = text.split()
        return " ".join([self.slang_dict.get(w, w) for w in words])

    # Tokenization
    def tokenize(self, text):
        return nltk.word_tokenize(text)

    # Stopword removal
    def remove_stopwords(self, tokens):
        return [w for w in tokens if w not in self.stop_words]

    # Lemmatization
    def lemmatize(self, tokens):
        doc = nlp(" ".join(tokens))
        return [token.lemma_ for token in doc]

    # Emoji handling
    def handle_emojis(self, text):
        return emoji.demojize(text)

    # Negation handling
    def handle_negation(self, tokens):
        result = []
        negate = False

        for word in tokens:
            if word in ["not", "no", "never"]:
                negate = True
                result.append(word)
            elif negate:
                result.append("not_" + word)
                negate = False
            else:
                result.append(word)

        return result

    # Spell correction (slow)
    def correct_spelling(self, text):
        return str(TextBlob(text).correct())

    # 🔥 MAIN PIPELINE
    def process(self, text):

        if self.config["lower"]:
            text = self.to_lower(text)

        if self.config["emoji"]:
            text = self.handle_emojis(text)

        if self.config["url"]:
            text = self.remove_urls(text)

        if self.config["repeat"]:
            text = self.normalize_repeated_chars(text)

        if self.config["slang"]:
            text = self.normalize_slang(text)

        if self.config["special_char"]:
            text = self.remove_special_chars(text)

        if self.config["spell"]:
            text = self.correct_spelling(text)

        if self.config["tokenize"]:
            tokens = self.tokenize(text)
        else:
            tokens = text.split()

        if self.config["negation"]:
            tokens = self.handle_negation(tokens)

        if self.config["stopwords"]:
            tokens = self.remove_stopwords(tokens)

        if self.config["lemmatize"]:
            tokens = self.lemmatize(tokens)

        return tokens

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 18.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 112.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [3]:
processor = AdvancedNLPPreprocessor()

text = "OMG broooo!!! I do not like this movie 😡😡 but story was goooood lol @user #movie"

result = processor.process(text)

print("Processed Output:")
print(result)

Processed Output:
['oh', 'god', 'broo', 'not_like', 'movie', 'enragedfaceenragedface', 'story', 'good', 'laugh', 'user', 'movie']
